## RAG Pipelines - Data Ingestion to Vector DB Pipeline

In [ ]:
import os
from langchain_community.document_loaders import DirectoryLoader,PyMuPDFLoader,PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path



In [14]:
from pathlib import Path
### Read all the Pdf's inside the dorectory

def process_all_pdfs(pdf_directory):
    """ Process all PDF files in a directory"""
    all_documents=[]
    pdf_dir = Path(pdf_directory)
    print(f"pdf_dir = {pdf_dir}")
    pdf_files =  list(pdf_dir.glob("**/*.pdf"))

    print(f"found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source Info to the Metadata
            for doc in documents:
                doc.metadata["source_file"]=pdf_file.name
                doc.metadata["file_type"]='pdf'
            
            all_documents.extend(documents)
            print(f" Loaded {len(documents)} pages")

        except Exception as e:
            print(f" Error:{e}")
        
    
    print(f"\nTotal Documents loaded: {len(all_documents)}")
    return all_documents

all_pdf_documents = process_all_pdfs("./files/pdffiles")

pdf_dir = files/pdffiles
found 3 PDF files to process

Processing: ALOK_SAHOO_CV (1).pdf
 Loaded 1 pages

Processing: ALOK-SAHOO-CV(old).pdf
 Loaded 1 pages

Processing: ALOK_SAHOO_RESUME(CV).pdf
 Loaded 2 pages

Total Documents loaded: 4


In [15]:
### Text Splitting get into chunks

def split_document(documents,chunk_size=1000,chunk_overlap=200):
    """ split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n","\n"," ",""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f" Split {len(documents)} documents into {len(split_docs)} chunks")

    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs


In [16]:
chunks = split_document(all_pdf_documents)

 Split 4 documents into 21 chunks

Example chunk:
Content: Alok Sahoo
Bhubaneswar | ✉ aloksahooas07@gmail.com | ♂phone +91-9348686594
/linkedin linkedin.com/in/alok-sahoo-as07 | /github github.com/Alok-as07
Professional Summary
Backend Developer with expertis...
Metadata: {'producer': 'PDFium', 'creator': 'PDFium', 'creationdate': '', 'source': 'files/pdffiles/ALOK_SAHOO_CV (1).pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'ALOK_SAHOO_CV (1).pdf', 'file_type': 'pdf'}


## Embedding and Vector Store DB

In [17]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List,Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity



In [21]:
class EmbeddingManager:
    """ Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self,model_name:str="all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager

        Args:
            model_name: HuggingFace model name for sentense embeddings
        """

        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """ load the SentenceTransformer model """
        try:
            print(f"Loading Embedding model:{self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f" Error loading model { self.model_name}: {e}")
            raise

    def generate_embedding(self,texts:List[str])-> np.ndarray:
        """
        Generate embedding for a list of texts

        Args:
            texts: List of text strings to embed
        
        Returns:
            numpy arrau opf embeddings with shape (len(texts),embedding_dim)
        """

        if not self.model:
            raise ValueError("model not found")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts,show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## Initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading Embedding model:all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5566.11it/s]


Model loaded successfully. Embedding dimension: 384


In [ ]:
### Vector Store

class VectorStore:
    """ Manages documents embeddings in a ChromaDB vector store"""

    def __init__(self,collection_name:str="pdf_documents",persist_directory:str="./data/vector_store"):
        """
        Initialize the vector store

        Args:
            collection_name : Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store

        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.colletion = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            #get or create Collection
            self.colletion = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description":"PDF document embeddings for RAG", "hnsw:space": "cosine"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existinng document in Collection: {self.colletion.count()}")
        except Exception as e:
            print(f" Error initializing vector store: {e}")
            raise


    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store

        Args:
        documents: List of LangChain documents
        embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to vector store...")

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text=[]
        embeddings_list=[]

        for i,(doc, embedding) in enumerate(zip(documents,embeddings)):
            #Generate unique Id
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            #prepare Metadata
            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # Document content
            documents_text.append(doc.page_content)

            # Embeddings
            embeddings_list.append(embedding.tolist())


        #Add to collection
        try:
            self.colletion.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f" Successfully added {len(documents)} documents to the vector store")
            print(f" Total documents in collection: { self.colletion.count()}")

        except Exception as e:
            print(f" Error adding documents to vector store: {e}")
            raise
        

            
## Initialize the Vector store
vectorStore = VectorStore()
vectorStore


Vector store initialized. Collection: pdf_documents
Existinng document in Collection: 0


In [25]:
### Convert the Text to Embeddings
texts=[doc.page_content for doc in chunks]

##Generate the embeddings
embeddings = embedding_manager.generate_embedding(texts)

## store in Vector DB
vectorStore.add_documents(chunks,embeddings)

Generating embeddings for 21 texts...


Batches: 100%|██████████| 1/1 [00:01<00:00,  1.92s/it]

Generated embeddings with shape: (21, 384)
Adding 21 documents to vector store...
 Successfully added 21 documents to the vector store
 Total documents in collection: 21


## Retriever Pipeline From VectorStore

In [ ]:
from google.cloud import bigquery
from google.cloud import bigquery
from huggingface_hub.inference._generated.types import zero_shot_image_classification
class RAGRetriever:
    """ Handles query based retrieval form the vector store"""

    def __init__(self,vector_store:VectorStore,embedding_manager= EmbeddingManager):
        """
       Initialize the retriever

        Args:
        vector_store: Vector store containing document embeddings
        embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager 

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query

        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold

        Returns:
            List of dictionaries containing retrieved documents and metadata

        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        #Generate query embedding
        query_embedding = self.embedding_manager.generate_embedding([query])[0]

        #search in vector store
        try:
            results = self.vector_store.colletion.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            #Process Results
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i,(doc_id,document,metadata,distance) in enumerate(zip(ids,documents,metadatas,distances)):
                    # Convert distance to similarity score ( ChromaDB used consine distance)
                    
                    # similarity_score = 1-distance
                    similarity_score = 1-(distance/2)  # this is for distace getting high and not able to retrieve
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id':doc_id,
                            'content':document,
                            'metadata':metadata,
                            'similarity_score':similarity_score,
                            'distance':distance,
                            'rank':i+1
                        })

                print(f" Retrieved {len(retrieved_docs)} documents ( After Filtering )")
            else:
                print("No document found")

            return retrieved_docs

        except Exception as e:
            print(f" Error during retrieval: {e}")
            return []

rag_retriever = RAGRetriever(vectorStore,embedding_manager)
rag_retriever


In [52]:
rag_retriever.retrieve("Professional summary")

Retrieving documents for query: 'Professional summary'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.21it/s]

Generated embeddings with shape: (1, 384)
 Retrieved 5 documents ( After Filtering )


[{'id': 'doc_4b852bd1_13',
  'content': 'summary\n \ngeneration\n \nfrom\n \ndoctor-patient\n \nconversations\n \nStructured\n \nData\n \nExtraction:\n \nConverting\n \nAI-generated\n \nconversational\n \nsummaries\n \ninto\n \nstructured,\n \nvalidated\n \nmedical\n \ndata\n \nAI\n \nService\n \nDelivery:\n \nREST\n \nAPI\n \ndesign\n \nfor\n \nAI\n \nconsumption,\n \nasync\n \nprocessing,\n \nJSON\n \nresponse\n \nhandling,\n \nerror\n \nhandling,\n \ndeployment,\n \nmonitoring,\n \nand\n \ntroubleshooting\n \nof\n \nAI\n \nservices\n \nin\n \nproduction\n \nPROFESSIONAL\n \nEXPERIENCE\n \nSoftware\n \nEngineer\n \n(SD-2)\n  \n|\n  \nSqubix\n \nDigital\n \nPvt\n \nLtd\n \n—\n \nBhubaneswar,\n \nIN\n2023\n \nOct\n \n–\n \nPresent\n \n•\n \nArchitected\n \n10+\n \nproduction\n \nmicroservices\n \nusing\n \nNestJS,\n \nimproving\n \nmodularity\n \nand\n \nreducing\n \ninter-service\n \ncoupling\n \nacross\n \nthe\n \nhealthcare\n \nSaaS\n \nplatform.\n \n•\n \nEngineered\n \nAI-powered\

# Integration vector DB Context pipeline With LLM output

In [54]:
### Simple RAG pipeline with groq LLM

from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
llm = ChatGroq(groq_api_key=os.environ["GROQ_API_KEY"],model="qwen/qwen3.8-27b",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results = retriever.retrieve(query,top_k=top_k)
    context = "\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return " No relevent context found to answer the question."

    ## generate the answer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely
        Context:
        {context}

        Question:{query}

        Answer:"""
    
    response = llm.invoke([prompt.format(context=context,query=query)])
    return response.content


In [58]:
answer = rag_simple("Who is Alok Sahoo",rag_retriever,llm)
print(f"\n\n{answer}")

Retrieving documents for query: 'Who is Alok Sahoo'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.97it/s]


Generated embeddings with shape: (1, 384)
 Retrieved 3 documents ( After Filtering )


Alok Sahoo is a Senior Backend Engineer and DevOps practitioner based in Bhubaneswar, India, with over 3 years of experience. He specializes in building scalable microservices, event-driven platforms, and cloud-native infrastructure for healthcare SaaS, with deep expertise in FHIR R4/HL7 interoperability, AI/LLM integration, and CI/CD automation across AWS, Docker, and Kubernetes.


## Enhanced RAG pipeline Features

In [59]:

### Enhanced RAG Pipeline Features
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
         - Returns answer, sources, confidence score, and optionally full context.
    """

    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}

    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [
        {
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:120] + '...'
        } for doc in results
    ]
    confidence = max([doc['similarity_score'] for doc in results])

    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.
    Context: {context}
    Question: {query}
    Answer: """
    response = llm.invoke([prompt.format(context=context,query=query)])

    output ={
        'answer':response.content,
        'sources':sources,
        'confidence':confidence
    }
    if return_context:
        output['context'] = context
    
    return output


# Example usage:
result = rag_advanced("Who is Alok Sahoo?", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])


Retrieving documents for query: 'Who is Alok Sahoo?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.17it/s]


Generated embeddings with shape: (1, 384)
 Retrieved 3 documents ( After Filtering )
Answer: Alok Sahoo is a Senior Backend Engineer and DevOps practitioner based in Bhubaneswar, India, with over three years of experience. He specializes in building scalable microservices, event-driven platforms, and cloud-native infrastructure for healthcare SaaS and interoperability. His key technical expertise includes FHIR R4/HL7 standards, AI/LLM integration, and CI/CD automation across AWS, Docker, and Kubernetes. Notable achievements include migrating over 500,000 healthcare records via Kafka, achieving 99.9% uptime on AWS Lambda, and reducing deployment cycles from approximately two hours to 15 minutes.
Sources: [{'source': 'ALOK-SAHOO-CV(old).pdf', 'page': 0, 'score': 0.33369189500808716, 'preview': 'ALOK SAHOO\nSenior Backend Engineer | DevOps & Cloud Infrastructure | Healthcare SaaS & Interoperability\nBhubaneswar, Ind...'}, {'source': 'ALOK_SAHOO_CV (1).pdf', 'page': 0, 'score': 0.326001644

In [ ]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("Professinoaly Alok is what ?", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'Professinoaly Alok is what ?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.06it/s]

Generated embeddings with shape: (1, 384)
 Retrieved 3 documents ( After Filtering )
Streaming answer:
Use the following context to answer the question concisely.
Context:
Alok Sahoo
Bhubaneswar | ✉ aloksahooas07@gmail.com | ♂phone +91-9348686594
/linkedin linkedin.com/in/alok-sahoo-as07 | /github github.com/Alok-as07
Professional Summary
Backend Developer with expertise in the MERN stack (MongoDB, Express.js, React.js,

 Node.js), A WS Cloud
services, and Kafka-based event-driven systems. Proven track record in developing scalable microservices, designing
secure APIs, and handling real-time healthcare data integration. Adept at leveraging serverless technologies for
efficiency and availability.
Skills
• Languages: Python, Java, JavaScript, TypeScript, Node.js, HTML, CSS
• F rameworks:NestJS, Express.js, React.js, Serverless, Terraform, Mocha, Jest, nginx, Kafka, Redis
• Databases: MongoDB, DynamoDB, MySQL, PostgreSQL
• Cloud Services: A WS (Lambda, EC2, ECR, Cognito, S3, IAM, CloudWatch, RDS, API Gateway), GCP,
Docker, Kubernetes
• DevOps: Jenkins, GitHub Actions, A WS CodePipeline
• V ersion Control:Git, GitHub, Bitbucket

ALOK SAHOO
Senior Backend Engineer | DevOps & Cloud Infrastructure | Healthcare SaaS & Interoperability
Bhubaneswar, India | +91-9348686594 | aloksahooas07@gmail.com | LinkedIn | GitHub
Professional Summary
Backend Engineer and DevOps practitioner with 3+ years building scalable mi